In [19]:
!pip -q install pyreadstat
import os
import numpy as np
import pandas as pd
import pyreadstat

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
PATH = f'{DATA_DIR}/HN24_ALL.sav'

print(os.path.exists('/content/drive/MyDrive'))
print(os.path.exists(DATA_DIR))
print(os.path.exists(PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True
True


In [20]:
import numpy as np
import pandas as pd

df, meta = pyreadstat.read_sav(PATH, usecols=['age', 'sex', 'BP1', 'mh_stress'])   # 4단계에서 공변량을 더함

df['stress'] = np.select([df['BP1'].isin([1, 2]), df['BP1'].isin([3, 4])], [1, 0], default=np.nan)

print(pd.crosstab(df['BP1'], df['stress'], dropna=False))       # 원 코드가 맞게 묶였는지
print(pd.crosstab(df['stress'], df['mh_stress'], dropna=False)) # mh_stress와 같은지

stress   0.0   1.0  NaN
BP1                    
1.0        0   297    0
2.0        0  1216    0
3.0     3409     0    0
4.0     1286     0    0
8.0        0     0  561
9.0        0     0  219
NaN        0     0    9
mh_stress   0.0   1.0  NaN
stress                    
0.0        4695     0    0
1.0           0  1513    0
NaN           0     0  789


In [15]:
# 전체 변수를 가진 원본 데이터 다시 불러오기
df, meta = pyreadstat.read_sav(PATH)

In [16]:
# 1단계 출력 및 변수 목록에서 확인한 실제 변수명으로 매핑
WD = dict(
    bed_h='BP16_11', bed_m='BP16_12', wake_h='BP16_13', wake_m='BP16_14'
)  # 주중 (잠자리에 든 시각 시간/분, 일어난 시각 시간/분)
WE = dict(
    bed_h='BP16_21', bed_m='BP16_22', wake_h='BP16_23', wake_m='BP16_24'
)  # 주말 (잠자리에 든 시각 시간/분, 일어난 시각 시간/분)
SPECIAL = [88, 99]  # 결측치나 해당 없음 처리할 특수문항 코드 (지침에 맞게 설정)                                                          # 확인 필요

def clean(col):
    return df[col].where(~df[col].isin(SPECIAL))

def sleep_hours(v):
    bed  = clean(v['bed_h'])  * 60 + clean(v['bed_m'])
    wake = clean(v['wake_h']) * 60 + clean(v['wake_m'])
    return ((wake - bed) % (24 * 60)) / 60      # 자정을 넘겨 잠들어도 맞게 계산

df['sleep_wd'] = sleep_hours(WD)
df['sleep_we'] = sleep_hours(WE)
df['sleep'] = (df['sleep_wd'] * 5 + df['sleep_we'] * 2) / 7

In [17]:
print(df[['sleep']].describe())                          # 최솟값·최댓값이 상식적인지
print((df['sleep'] < 3).sum(), (df['sleep'] > 16).sum())  # 이상값 후보 수

             sleep
count  6202.000000
mean      7.430463
std       1.253988
min       2.500000
25%       6.678571
50%       7.500000
75%       8.142857
max      21.000000
4 1


In [18]:
# 1. 이상값(3시간 미만 또는 16시간 초과)을 제외한 유효 수면 시간 데이터 생성
df_sleep_valid = df[
    df['sleep'].notna() & (df['sleep'] >= 3) & (df['sleep'] <= 16)
].copy()

# 2. 제외된 인원 수 확인
excluded_outlier = (
    df['sleep'].notna() & ((df['sleep'] < 3) | (df['sleep'] > 16))
).sum()
print(f'• 수면 시간 이상값으로 제외된 인원: {excluded_outlier}명')

# 3. 이상값이 제외된 최종 수면 시간 기술통계 확인
print('\n=== [이상값 제외 후 수면 시간 기술통계] ===')
print(df_sleep_valid[['sleep']].describe())

• 수면 시간 이상값으로 제외된 인원: 5명

=== [이상값 제외 후 수면 시간 기술통계] ===
             sleep
count  6197.000000
mean      7.431363
std       1.236618
min       3.000000
25%       6.678571
50%       7.500000
75%       8.142857
max      13.857143
